In [ ]:
"""
import the vowpal wabbit library.
"""
pip install vowpalwabbit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 19.5 MB/s eta 0:00:00


In [ ]:
"""
import the other required libraries.
"""
import pandas as pd
import vowpalwabbit
import numpy as np
import json

model = 'bert'

In [ ]:
"""
load the genre_mapping dictioanry from the file directory. The genre_mapping dictionary is created once and used for all the three models. It contains the numerical representation of genres.
"""
genres_mappings = dict()
txt_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/input/genre_mapping.txt'
with open(txt_file_path, 'r') as txt_file:
    genres_mappings = json.loads(txt_file.read())
print(genres_mappings)

unique_genres = list(genres_mappings.values())
print(unique_genres)

{'Crime': 1, 'Western': 2, 'Fantasy': 3, 'Science Fiction': 4, 'Comedy': 5, 'Family': 6, 'Mystery': 7, 'Drama': 8, 'Animation': 9, 'Thriller': 10, 'Documentary': 11, 'Horror': 12, 'Romance': 13, 'TV Movie': 14, 'History': 15, 'War': 16, 'Adventure': 17, 'Music': 18, 'Action': 19}
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [ ]:
"""
load the dataset to be used for the experiment. This dataset was created by the data prepare step. This will act as the main dataset for this experiment also referred as the duplicate_instances_dataset in the report and will be refered as action_context_map for the experiment code.
"""
action_context_map = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_action_context_map.csv')

In [ ]:
"""
convert the textual genres to the numerical genres columns.
"""
from ast import literal_eval
action_context_map['genres_indices'] = action_context_map['genres_indices'].apply(literal_eval)

In [ ]:
"""
print and see the records of action_context_map.
"""
action_context_map.head(100)

,genres_index,keywords_embedding_string,genres_indices,id,original_title_embedding_string,overview_embedding_string,release_date
0,1,2.24766999e-01 1.48856416e-01 -4.26082835e-0...,"[11, 8]",1114,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...,3.44745591e-02 -2.66509261e-02 -2.78230965e-01...,6/29/2003
1,1,8.62499252e-02 6.64695725e-02 4.50119734e-0...,[8],1115,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...,-3.16557437e-01 -1.12028293e-01 2.61700094e-0...,9/15/2005
2,1,2.55626798e-01 3.36861275e-02 -1.86209410e-0...,"[8, 16]",1116,-1.84306309e-01 7.14582950e-02 -1.47662491e-0...,-1.01136873e-02 1.58037364e-01 1.00762844e-0...,6/23/2006
3,1,-1.35598034e-01 -3.07193458e-01 -3.09346825e-0...,"[19, 8, 10]",1123,4.00660694e-01 -1.56783655e-01 -1.28297895e-0...,-1.07681625e-01 -8.75450373e-02 -9.33825895e-0...,10/27/2006
4,1,3.90980601e-01 2.90082335e-01 5.26184797e-0...,"[8, 7, 4]",1124,3.64068747e-02 -1.96709305e-01 -4.71620530e-0...,1.85870528e-01 1.91051021e-01 5.43662846e-01...,10/19/2006
...,...,...,...,...,...,...,...
95,1,-1.56553015e-01 2.18778942e-02 -6.90055415e-0...,"[13, 8]",1413,-2.47678280e-01 -2.72932470e-01 -2.96154052e-0...,-4.93864447e-01 4.43464778e-02 1.31124616e-0...,10/1/1993
96,1,1.60582602e-01 2.75252998e-01 4.82334405e-0...,"[8, 7, 13]",1414,-2.45494217e-01 -3.40429917e-02 -4.39187050e-0...,1.72036439e-02 6.22920170e-02 4.54654008e-01...,9/4/1998
97,1,8.97325203e-03 1.06297635e-01 4.33871061e-0...,"[5, 8, 1]",1415,3.13995600e-01 -4.59249973e-01 9.57270935e-0...,1.11045711e-01 -2.52095312e-01 4.37954247e-01...,9/5/2003
98,1,1.64257735e-01 2.46206354e-02 1.07073337e-0...,"[8, 16]",1416,-6.13639355e-02 4.22465712e-01 3.59010249e-0...,-3.67381349e-02 -9.45226327e-02 5.32655194e-0...,11/22/2002


In [ ]:
"""
print and see the size of action_context_map
"""
action_context_map.shape[0]

190000

In [ ]:
"""
a variable to ensure test dataset is selected at random from the entire dataset.
"""
random_test_data_flag = True

In [ ]:
"""
the test dataset will always be the random test dataset. But incase if the experiment needs to be performed for a test data for which the model has prior knowledge, the set the flag to False in the above line.
"""
if random_test_data_flag:
  test_movies_df = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_RANDOM_TEST_file.csv')
else:
  test_movies_df = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_TEST_file.csv')

test_movies_df['genres_indices'] = test_movies_df['genres_indices'].apply(literal_eval)

In [ ]:
"""
print and see the records of test dataset.
"""
test_movies_df.head(10)

,keywords_embedding_string,genres_indices,id,original_title_embedding_string,release_date,overview_embedding_string
0,4.54344675e-02 1.17894545e-01 -3.63955915e-0...,[11],1223382,-1.47495717e-02 -5.30206919e-01 -1.88464835e-0...,11/3/2023,-2.13244721e-01 2.55034894e-01 -6.37503574e-0...
1,3.19913626e-01 -5.27107537e-01 -2.63918757e-0...,[5],583413,2.75947928e-01 2.41989881e-01 3.55889231e-0...,9/29/1984,7.29451329e-02 -1.12443054e-02 2.96463192e-01...
2,-2.09423900e-02 -2.93477088e-01 -1.62958860e-0...,[11],342852,-2.09684577e-02 -9.43729877e-02 -3.63037705e-0...,1/1/1968,-9.82245803e-02 8.45943764e-02 -1.99082881e-0...
3,3.01734626e-01 3.16437602e-01 1.29360169e-0...,"[8, 5, 13]",10401,-7.22050786e-01 4.38166745e-02 5.56653738e-0...,3/31/1999,-5.80361001e-02 -1.37145236e-01 3.56253237e-0...
4,1.35435790e-01 3.52852523e-01 -7.12127328e-0...,[11],802160,1.94779679e-01 -5.69366552e-02 -2.45968461e-0...,10/20/2000,3.49366777e-02 2.46183172e-01 9.22734365e-02...
5,3.05170715e-01 -1.86433837e-01 -3.91963050e-0...,"[4, 12]",1016490,2.00182781e-01 -3.43232036e-01 1.50550380e-0...,NaN,1.23634212e-01 -7.09160194e-02 7.37648830e-02...
6,3.63106728e-01 3.86320204e-02 -1.81825995e-0...,[11],331916,-2.46388704e-01 -1.44056961e-01 -1.27164826e-0...,9/29/2012,6.95594549e-01 5.15481234e-01 3.81713241e-01...
7,3.66826020e-02 1.06648088e-01 1.36117324e-0...,[11],45558,3.49452198e-01 3.43854865e-03 -1.74925864e-0...,10/14/1992,-2.61206995e-03 3.09144687e-02 1.69136092e-0...
8,-1.19429650e-02 -1.30293936e-01 2.18185961e-0...,"[12, 7]",37726,-4.02784020e-01 1.45995915e-01 -3.79183292e-0...,10/6/1998,-2.35990673e-01 -1.18590377e-01 1.46030664e-0...
9,3.78917456e-01 -1.36297587e-02 -5.32229841e-0...,[8],1142678,-2.23415736e-02 -5.52823126e-01 1.50125679e-0...,3/2/2019,1.42520845e-01 2.16124095e-02 2.21767262e-01...


In [ ]:
"""
print and see the size of action_context_map
"""
test_movies_df.shape[0]

38000

In [ ]:
"""
load the train dataset to be used for the experiment. This dataset was created by the data prepare step.
"""
train_movies_df = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_TRAIN_file.csv')
train_movies_df['genres_indices'] = train_movies_df['genres_indices'].apply(literal_eval)

In [ ]:
"""
print and see the records of train dataset.
"""
train_movies_df.head(10)

,genres_index,keywords_embedding_string,genres_indices,id,original_title_embedding_string,overview_embedding_string,release_date,rewards,cost
0,1,2.99294163e-02 4.77882400e-02 5.39655447e-0...,"[6, 9, 13, 3]",14128,1.54227987e-01 -2.74204910e-02 8.75207186e-0...,-1.80847988e-01 9.59640592e-02 2.71085948e-0...,2/23/2002,0.0,1.0
1,5,7.27944076e-02 1.54586062e-01 3.97218347e-0...,"[6, 3, 9, 14, 18, 7, 17]",13351,-1.61057934e-01 9.18286201e-03 5.20056114e-0...,-1.66192040e-01 2.83178777e-01 4.61630374e-0...,9/3/1994,0.0,1.0
2,18,1.78888366e-01 3.32385972e-02 3.66470397e-0...,"[8, 7, 10]",1126,-7.42457658e-02 7.33588934e-02 -1.90228760e-0...,-2.62416571e-01 8.42893571e-02 3.06196541e-0...,10/7/2004,0.0,1.0
3,1,4.08782005e-01 -2.44279161e-01 -1.25325501e-0...,"[5, 12, 3]",16846,1.77843377e-01 -5.62943220e-02 -9.54189803e-0...,4.42602038e-02 2.47185454e-01 5.65411150e-01...,7/4/2008,0.0,1.0
4,3,-2.32977316e-01 4.70709056e-02 7.09981546e-0...,"[19, 5, 1]",13534,1.83639139e-01 -1.54455170e-01 -6.65129781e-0...,-7.43860379e-02 -1.71329945e-01 2.39840453e-0...,1/25/2007,0.0,1.0
5,4,2.86174089e-01 -4.94969487e-01 2.01115936e-0...,"[19, 1, 10]",14398,2.69247055e-01 -7.78893009e-02 -6.88501835e-0...,2.60788828e-01 7.45475367e-02 1.48685470e-01...,10/8/1997,0.0,1.0
6,2,2.77489662e-01 -1.46328017e-01 1.12790681e-0...,"[12, 4, 7, 8]",9682,9.52914581e-02 3.90896648e-02 -5.01335971e-0...,5.63859284e-01 6.54903591e-01 4.19694841e-01...,8/11/2006,0.0,1.0
7,19,1.47977665e-01 5.86288690e-04 1.69191569e-0...,"[5, 1]",11872,-0.78046966 -0.29745832 0.1540196 -0.3014274...,-3.09160084e-01 3.92666534e-02 2.95196921e-0...,10/19/1962,0.0,1.0
8,5,3.75225157e-01 1.40544996e-01 3.87620479e-0...,"[10, 12]",18781,9.40434169e-03 -8.76316130e-02 -3.81292820e-0...,-5.77834919e-02 9.20622945e-02 1.85866728e-0...,3/27/2009,0.0,1.0
9,14,-1.90755799e-01 2.30410576e-01 1.29587382e-0...,"[19, 5, 8]",15836,-3.43865544e-01 -4.76595134e-01 -1.24916762e-0...,-2.98591107e-01 1.26263320e-01 3.62463653e-0...,3/9/2007,0.0,1.0


In [ ]:
"""
print and see the records of train dataset.
"""
train_movies_df.shape[0]

152000

In [ ]:
"""
print distribution of movie genres
"""
all_numbers = [num for sublist in train_movies_df['genres_indices'] for num in sublist]
counts_df = pd.DataFrame(all_numbers, columns=['Number']).groupby('Number').size().reset_index(name='Count')
print(counts_df)

    Number  Count
0        1  22191
1        2   3682
2        3  11751
3        4  13099
4        5  57886
5        6  13359
6        7  10038
7        8  69064
8        9   8420
9       10  30004
10      11   7081
11      12  17500
12      13  25720
13      14   4340
14      15   6433
15      16   5004
16      17  19320
17      18   6261
18      19  27299


In [ ]:
"""
load the vowpal wabbit formatted train dataset
"""
vw_train_data = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_vw_train_data.txt')

In [ ]:
"""
load the vowpal wabbit model and train it on train data
"""
vw_model = vowpalwabbit.Workspace(f"--cb_explore {len(unique_genres)} --quiet --epsilon 0.1")
for example in vw_train_data:
    vw_model.learn(example)

In [ ]:
"""
save the trained model
"""
vw_model.save(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_cb_EP-0.1.model')

In [ ]:
"""
this function assigns the cost to the predicted action
"""
def test_assign_cost(predicted_action, movie_id):
    action_context_map = test_movies_df
    search_col = 'id'
    search_val = int(movie_id)
    target_column = 'genres_indices'
    res = action_context_map[(action_context_map[search_col] == search_val)]
# uses the same mathematical formula as stated in the report equation 7
    ind = -1
    index_list = res[target_column].iloc[0]
    if predicted_action in index_list:
      ind = index_list.index(predicted_action)
      if ind == 0:
          return 0
      elif ind >= 1 :
        return ind/len(index_list)
    else:
      return 1

In [ ]:
"""
this function is the probability mass function or pmf.  It takes a list pmf as input, which represents the probabilities of each outcome.
"""
import random
def sample_custom_pmf(pmf):
    total = sum(pmf)
    scale = 1 / total
    pmf = [x * scale for x in pmf]
    draw = random.random()
    sum_prob = 0.0
    for index, prob in enumerate(pmf):
        sum_prob += prob
        if sum_prob > draw:
            return index, prob

In [ ]:
"""
this function is to define the prediction in the model. It passes the action, cost and context in the vowpal wabbit format and triggers the model to make predictions
"""
def predict_genre(row):
    formatted_row = row["keywords_embedding_string"]
    formatted_id = row["id"]
    formatted_original_title_embedding_string = row["original_title_embedding_string"]
    formatted_release_date = row["release_date"]
    formatted_overview_embedding_string = row["overview_embedding_string"]
    context_keywords = f'| feature1:{formatted_id} feature2:{formatted_row} feature3:{formatted_original_title_embedding_string} feature4:{formatted_release_date} feature5:{formatted_overview_embedding_string}'
    action_probabilities = vw_model.predict(context_keywords)
    chosen_index, prob = sample_custom_pmf(action_probabilities)
    return (chosen_index + 1, prob, action_probabilities)

In [ ]:
"""
this function is used to run the prediction iterations.
It stores the results of each iterations in specified files like rewards, regret, accuracy, conversion rate
"""
num_iterations = 300
res = []

totalObtainedRewardsSum = 0
totalObtainedCostSum = 0
iterationRewardsCounter = 0
iterationCostCounter = 0
iterationRewardsList = []
iterationCostList = []
actionCounts = {}
successCounts = {}
accurateCounter = 0


for iteration in range(num_iterations):
  shuffled_indices = np.random.permutation(len(test_movies_df))
  for index in shuffled_indices:
    row = test_movies_df.iloc[index]
    resl = predict_genre(row)
    predicted_genre, prob = resl[0], resl[1]
    keywords_string = row['keywords_embedding_string']
    movie_id = row['id']
    original_title_embedding_string = row["original_title_embedding_string"]
    release_date = row["release_date"]
    overview_embedding_string = row["overview_embedding_string"]
    actual_cost = test_assign_cost(predicted_genre, movie_id)
    totalObtainedRewardsSum += (1-actual_cost)
    iterationRewardsCounter += (1-actual_cost)
    totalObtainedCostSum += actual_cost
    iterationCostCounter += actual_cost

    if predicted_genre not in actionCounts:
      actionCounts[predicted_genre] = 0
    actionCounts[predicted_genre] += 1

    if predicted_genre in row["genres_indices"]:
      accurateCounter += 1
      if predicted_genre not in successCounts:
        successCounts[predicted_genre] = 0
      successCounts[predicted_genre] += 1

    vw_model.learn(f'{predicted_genre}:{actual_cost}:{prob} | feature1:{movie_id} feature2:{keywords_string} feature3:{original_title_embedding_string} feature4:{release_date} feature5:{overview_embedding_string}')

  iterationRewardsList.append(iterationRewardsCounter)
  iterationCostList.append(iterationCostCounter)
  iterationRewardsCounter = 0
  iterationCostCounter = 0

conversion_rates = {}
for action, success_count in successCounts.items():
    total_count = actionCounts[action]
    conversion_rate = success_count / total_count
    conversion_rates[action] = conversion_rate

modelAccuracy = accurateCounter / (len(test_movies_df) * num_iterations)

In [ ]:
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_res_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    for item in res:
        file.write("%s\n" % item)

In [ ]:
"""
print the cumulative rewards over 300 iterations
"""
print(totalObtainedRewardsSum)

6345474.973026555


In [ ]:
"""
print the cumulative regret over 300 iterations
"""
print(totalObtainedCostSum)

5054525.02698537


In [ ]:
"""
print the cumulative rewards after end of each iteration
"""
print(iterationRewardsList)

[9721.472222222223, 12571.455555555516, 13667.538888888801, 14252.005555555508, 14624.057936507854, 15001.341269841207, 15298.366666666572, 15578.869841269745, 15759.78412698401, 16068.383333333224, 16193.32460317449, 16310.445238095124, 16465.81388888877, 16703.415079364975, 16909.394841269757, 17020.196428571322, 17043.277380952255, 17217.655555555455, 17307.67142857132, 17454.805555555417, 17570.730952380865, 17709.257142857077, 17817.916666666584, 17983.66666666659, 17985.546031745933, 18079.017460317373, 18068.32857142848, 18314.34523809516, 18305.502380952265, 18301.63571428567, 18459.31904761897, 18545.888095238057, 18605.888095238064, 18781.39523809517, 18655.619047619024, 18692.29761904759, 18870.602380952285, 18807.74404761896, 18940.42857142853, 18943.77976190469, 18856.104761904768, 19072.021428571377, 19134.116666666632, 19265.526984126962, 19253.779365079343, 19270.278174603154, 19312.32222222217, 19436.79126984126, 19437.726984126974, 19429.996031745977, 19520.6555555555

In [ ]:
"""
print the cumulative regret after end of each iteration
"""
print(iterationCostList)

[28278.52777777773, 25428.544444444367, 24332.461111111006, 23747.994444444408, 23375.942063491988, 22998.65873015874, 22701.63333333332, 22421.130158730168, 22240.21587301581, 21931.61666666659, 21806.675396825307, 21689.55476190472, 21534.186111111103, 21296.584920634872, 21090.605158730177, 20979.80357142852, 20956.722619047607, 20782.344444444432, 20692.328571428563, 20545.19444444452, 20429.26904761901, 20290.742857142817, 20182.083333333303, 20016.333333333358, 20014.45396825402, 19920.982539682558, 19931.671428571408, 19685.654761904745, 19694.49761904771, 19698.364285714288, 19540.680952380957, 19454.111904761907, 19394.111904761885, 19218.604761904804, 19344.380952380947, 19307.70238095239, 19129.3976190477, 19192.255952381027, 19059.57142857147, 19056.220238095306, 19143.89523809521, 18927.978571428637, 18865.883333333368, 18734.47301587307, 18746.220634920664, 18729.72182539688, 18687.677777777863, 18563.208730158767, 18562.273015873106, 18570.003968254037, 18479.34444444453

In [ ]:
"""
save the cumulative rewards data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_reward_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    for item in iterationRewardsList:
        file.write("%s\n" % item)

In [ ]:
"""
save the cumulative regret data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_cost_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    for item in iterationCostList:
        file.write("%s\n" % item)

In [ ]:
"""
save the conversion data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_conversion_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
  for action, conversion_rate in conversion_rates.items():
    file.write("%s %s\n" % (action, conversion_rate))

In [ ]:
"""
save the accuracy data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_accuracy_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    file.write("%s \n" % modelAccuracy)

In [ ]:
# End